# HyperSense v1.2 — Stage 3: Clinical Enrichment

Stage 3 extends the HyperSense project from the established parsimonious hypertension-risk model to a clinically enriched, Nigeria-specific model using the REMAH cohort.

Stages 1 and 2 evaluated transportability and regional adaptation of the parsimonious six-predictor model. Stage 3 addresses a different question: whether additional clinically relevant anthropometric, behavioural, and laboratory variables improve hypertension risk prediction in the Nigerian population beyond the established model.

This analysis uses the Nigerian REMAH dataset only. Both models are developed and evaluated under the same analytical framework and differ only in their predictor sets:

- **Parsimonious model:** the established HyperSense six-predictor feature set, retrained on REMAH data.
- **Clinically enriched model:** the parsimonious feature set plus additional clinically relevant variables available from the REMAH dataset.

### Outcome

Hypertension is defined according to the published REMAH methodology (Odili et al., 2020) as measured systolic blood pressure ≥140 mmHg and/or diastolic blood pressure ≥90 mmHg and/or current use of antihypertensive medication.

Blood pressure is defined as the average of five consecutive clinic measurements, with all five measurements required. Current antihypertensive treatment is an independent component of the outcome definition; therefore, participants reporting current antihypertensive treatment are classified as hypertensive even when complete blood pressure measurements are unavailable.

`HTN_DR` (`Hypertensive drug therapy`) is used only in outcome construction and is excluded from the predictor sets.

$$
\text{HTN}_{\text{Stage3}} =
\begin{cases}
1, & \text{if complete mean SBP} \geq 140 \text{ or complete mean DBP} \geq 90 \text{ or HTN\_DR}=1 \\
0, & \text{if complete mean SBP}<140 \text{ and complete mean DBP}<90 \text{ and HTN\_DR}\neq1
\end{cases}
$$

### Modelling approach

The primary modelling approach is **XGBoost classification.**

Model development follows the methodological framework established in Stages 1 and 2, including a fixed stratified train/test split, training-only hyperparameter tuning, out-of-fold threshold selection using Youden's index, and evaluation of both discrimination and calibration.

Both models are trained and evaluated on the same analytical cohort and under the same modelling framework, allowing the effect of clinical enrichment to be assessed directly.

### Research question

> Does incorporating additional clinically relevant anthropometric, behavioural, and laboratory variables improve hypertension risk prediction in the Nigerian population beyond the parsimonious HyperSense model?

### Structure

1. Stage 3 Overview
2. Data Loading and Initial Inspection
3. Data Dictionary Mapping and Data Quality Audit
4. Outcome Definition
5. Feature Engineering
6. Missingness and Analytical Cohort
7. Predictor Definition and Leakage Assessment
8. Train/Test Split
9. Parsimonious Model
10. Clinically Enriched Model
11. Hyperparameter Tuning
12. Threshold Selection
13. Final Model Evaluation
14. Parsimonious vs. Enriched Comparison
15. SHAP Explainability
16. Export Results for Reproducibility
17. Stage 3 Conclusion

This structure keeps the major methodological decisions explicit before model development.

In [22]:
# Record package versions for reproducibility

import pandas as pd
import numpy as np
import sklearn
import xgboost

print(f"pandas:   {pd.__version__}")
print(f"numpy:    {np.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"xgboost:  {xgboost.__version__}")

pandas:   2.3.3
numpy:    2.4.0
scikit-learn: 1.8.0
xgboost:  3.3.0


In [23]:
# Core libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Scikit-learn
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
    classification_report,
    brier_score_loss,
    precision_score,
    recall_score,
    roc_curve,
)
from sklearn.calibration import calibration_curve

from sklearn.base import clone

# Calibration model for estimating calibration intercept and slope
from sklearn.linear_model import LogisticRegression

# XGBoost
from xgboost import XGBClassifier

from pathlib import Path
import json

# Reproducibility
RANDOM_STATE = 99

np.random.seed(RANDOM_STATE)

In [24]:
# Load the REMAH cohort dataset

nigeria = pd.read_excel("../data/REMAH/remah-1.xlsx")

print(f"Nigeria dataset: {len(nigeria):,} rows")

Nigeria dataset: 4,187 rows


In [25]:
# Check all column names
print(nigeria.columns.tolist())

['S_NO', 'PIN', 'SEX', 'EST_AGE', 'DOB', 'AGE', 'SITE', 'EDU_QUA', 'M_STATUS', 'W_STATUS', 'SMK_C', 'SMK_D', 'SMK_D1', 'SMK_D2', 'M_CGRT', 'HR_CGRT', 'PIPE', 'CIGAR', 'SMK_OTHERS', 'SMK_P', 'SMK_P1', 'SMK_P2', 'SMKL', 'SMKL_MO', 'SMKL_NO', 'SMKL_CHEW', 'SMKL_OTHERS', 'SMK_PA_H', 'SMK_PA_WO', 'DRK', 'DRK_12', 'DRK12_FREQ', 'DRK_30', 'WRK_VI', 'WRK_VI_D', 'WRK_VI_T', 'WRK_MI', 'WRK_MI_D', 'WRK_MI_T', 'PA_W', 'PA_W_D', 'PA_W_T', 'SPRT_VI', 'SPRT_VI_D', 'SPRT_VI_T', 'SPRT_MI', 'SPRT_MI_D', 'SPRT_MI_T', 'HTN', 'HTN_DR', 'DM_M', 'DM', 'DM_INS', 'DM_OHA', 'CSBP1', 'CDBP1', 'CSBP2', 'CDBP2', 'CSBP3', 'CDBP3', 'CSBP4', 'CDBP4', 'CSBP5', 'CDBP5', 'PR', 'HT', 'WT', 'WC', 'HC', 'BLD_GLU', 'NA_URINE', 'HDL_CHOL', 'LDL_CHOL', 'TAG', 'TO_CHOL', 'K_URINE']


#### 3. Data Dictionary Mapping and Data Quality Audit

Mapping the pre-specified predictors and outcome variables to the REMAH data dictionary, then assessing coding, missingness, and just basic data quality before feature construction.

In [30]:
# Raw variables required for Stage 3
required_vars = [
    # Outcome
    "HTN",
    "HTN_DR",

    # Blood pressure measurements 
    "CSBP1", "CSBP2", "CSBP3", "CSBP4", "CSBP5",
    "CDBP1", "CDBP2", "CDBP3", "CDBP4", "CDBP5",
    
    # Parsimonious predictors
    "AGE",
    "SEX",
    "EDU_QUA",
    "SITE",
    "SMK_C",
    "SMKL",
    "WT",
    "HT",

    # Enriched predictors
    "PR",
    "M_STATUS",
    "DM",
    "HDL_CHOL",
    "TO_CHOL",
    "LDL_CHOL",
    "SMK_P",
]

# Check which required variables are present
present = [col for col in required_vars if col in nigeria.columns]
missing = [col for col in required_vars if col not in nigeria.columns]

print(f"Present: {len(present)}/{len(required_vars)}")
print(f"Missing: {missing}")

Present: 27/27
Missing: []


In [36]:
# Inspect data types and missingness for Stage 3 variables

audit_vars = required_vars

audit = pd.DataFrame({
    "dtype": nigeria[audit_vars].dtypes.astype(str),
    "missing_n": nigeria[audit_vars].isna().sum(),
    "missing_pct": (nigeria[audit_vars].isna().mean() * 100).round(2),
    "unique_n": nigeria[audit_vars].nunique(dropna=True)
})

audit.sort_values("missing_pct", ascending=False)

,dtype,missing_n,missing_pct,unique_n
HTN_DR,float64,2976,71.08,2
DM,float64,1917,45.78,3
HTN,float64,972,23.21,2
TO_CHOL,float64,785,18.75,573
LDL_CHOL,float64,732,17.48,492
HDL_CHOL,float64,575,13.73,241
SMKL,float64,480,11.46,2
PR,float64,173,4.13,89
SMK_P,float64,135,3.22,2
SMK_C,float64,43,1.03,2


In [37]:
# Inspect categorical variable coding

categorical_vars = [
    "HTN",
    "HTN_DR",
    "SEX",
    "EDU_QUA",
    "SITE",
    "M_STATUS",
    "SMK_C",
    "SMKL",
    "SMK_P",
    "DM"
]

for col in categorical_vars:
    print(f"\n--- {col} ---")
    print(nigeria[col].value_counts(dropna=False).sort_index())


--- HTN ---
HTN
1.0    1030
2.0    2185
NaN     972
Name: count, dtype: int64

--- HTN_DR ---
HTN_DR
1.0     495
2.0     716
NaN    2976
Name: count, dtype: int64

--- SEX ---
SEX
1    1814
2    2373
Name: count, dtype: int64

--- EDU_QUA ---
EDU_QUA
1.0      658
2.0      193
3.0      707
4.0     1158
5.0     1131
7.0      299
55.0       1
88.0       1
NaN       39
Name: count, dtype: int64

--- SITE ---
SITE
1    2171
2    2016
Name: count, dtype: int64

--- M_STATUS ---
M_STATUS
1.0      922
2.0     2758
3.0       29
4.0       27
5.0      427
6.0        9
88.0       2
NaN       13
Name: count, dtype: int64

--- SMK_C ---
SMK_C
1.0     166
2.0    3978
NaN      43
Name: count, dtype: int64

--- SMKL ---
SMKL
1.0      98
2.0    3609
NaN     480
Name: count, dtype: int64

--- SMK_P ---
SMK_P
1.0     311
2.0    3741
NaN     135
Name: count, dtype: int64

--- DM ---
DM
1.0     208
2.0    2061
5.0       1
NaN    1917
Name: count, dtype: int64


In [38]:
# Blood pressure measurement completeness

bp_vars = [
    "CSBP1", "CSBP2", "CSBP3", "CSBP4", "CSBP5",
    "CDBP1", "CDBP2", "CDBP3", "CDBP4", "CDBP5"
]

bp_audit = pd.DataFrame({
    "missing_n": nigeria[bp_vars].isna().sum(),
    "missing_pct": (nigeria[bp_vars].isna().mean() * 100).round(2)
})

bp_audit

,missing_n,missing_pct
CSBP1,4,0.10
CSBP2,4,0.10
CSBP3,3,0.07
CSBP4,4,0.10
CSBP5,16,0.38
CDBP1,5,0.12
CDBP2,4,0.10
CDBP3,3,0.07
CDBP4,4,0.10
CDBP5,18,0.43


In [40]:
# Categorical codes supported by data dictionary

expected_codes = {
    "HTN": {1, 2},
    "HTN_DR": {1, 2},
    "SEX": {1, 2},
    "EDU_QUA": {1, 2, 3, 4, 5, 7, 88},
    "SITE": {1, 2},
    "M_STATUS": {1, 2, 3, 4, 5, 6, 88},
    "SMK_C": {1, 2},
    "SMKL": {1, 2},
    "SMK_P": {1, 2},
    "DM": {1, 2},
}

for col, valid_codes in expected_codes.items():
    observed = set(nigeria[col].dropna().unique())
    unexpected = sorted(observed - valid_codes)

    print(f"{col}:")
    print(f"  Observed: {sorted(observed)}")
    print(f"  Unexpected: {unexpected if unexpected else 'None'}")
    print()

HTN:
  Observed: [np.float64(1.0), np.float64(2.0)]
  Unexpected: None

HTN_DR:
  Observed: [np.float64(1.0), np.float64(2.0)]
  Unexpected: None

SEX:
  Observed: [np.int64(1), np.int64(2)]
  Unexpected: None

EDU_QUA:
  Observed: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(7.0), np.float64(55.0), np.float64(88.0)]
  Unexpected: [np.float64(55.0)]

SITE:
  Observed: [np.int64(1), np.int64(2)]
  Unexpected: None

M_STATUS:
  Observed: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(88.0)]
  Unexpected: None

SMK_C:
  Observed: [np.float64(1.0), np.float64(2.0)]
  Unexpected: None

SMKL:
  Observed: [np.float64(1.0), np.float64(2.0)]
  Unexpected: None

SMK_P:
  Observed: [np.float64(1.0), np.float64(2.0)]
  Unexpected: None

DM:
  Observed: [np.float64(1.0), np.float64(2.0), np.float64(5.0)]
  Unexpected: [np.float64(5.0)]



In [42]:
# Range and plausibility checks for continuous variables

continuous_vars = [
    "AGE",
    "HT",
    "WT",
    "PR",
    "HDL_CHOL",
    "TO_CHOL",
    "LDL_CHOL",
]

continuous_audit = nigeria[continuous_vars].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

continuous_audit["missing_n"] = nigeria[continuous_vars].isna().sum()
continuous_audit["missing_pct"] = (
    nigeria[continuous_vars].isna().mean() * 100
).round(2)

continuous_audit

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,missing_n,missing_pct
AGE,4183.0,43.835090,16.297721,14.509589,18.321315,20.484932,30.660274,41.958904,55.142466,72.950137,83.263726,98.791781,4,0.10
HT,4164.0,163.419476,8.792892,131.000000,145.000000,150.000000,157.000000,163.000000,169.125000,179.000000,185.000000,195.000000,23,0.55
WT,4163.0,65.348475,14.779346,30.000000,40.000000,45.000000,55.000000,63.000000,74.000000,93.000000,109.000000,136.000000,24,0.57
PR,4014.0,73.609367,12.657101,7.000000,50.000000,56.000000,65.000000,72.000000,81.000000,96.000000,110.870000,138.000000,173,4.13
HDL_CHOL,3612.0,0.812947,0.562132,0.000000,0.010000,0.160000,0.430000,0.750000,1.050000,1.894500,2.595600,10.100000,575,13.73
TO_CHOL,3402.0,3.205344,1.328088,0.010000,0.830000,1.170000,2.170000,3.190000,4.170000,5.379500,6.300000,9.000000,785,18.75
LDL_CHOL,3455.0,1.995386,1.154278,0.000000,0.090000,0.320000,1.040000,1.960000,2.810000,3.930000,4.799200,8.800000,732,17.48


In [43]:
# Inspect the observed ranges of the five clinic BP readings

bp_summary = nigeria[bp_vars].describe(
    percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
).T

bp_summary["missing_n"] = nigeria[bp_vars].isna().sum()
bp_summary["missing_pct"] = (
    nigeria[bp_vars].isna().mean() * 100
).round(2)

bp_summary

,count,mean,std,min,1%,5%,50%,95%,99%,max,missing_n,missing_pct
CSBP1,4183.0,127.035381,24.217771,60.0,90.0,96.0,122.0,172.0,200.0,250.0,4,0.10
CSBP2,4183.0,126.546498,23.868516,64.0,88.0,96.0,122.0,172.0,198.0,252.0,4,0.10
CSBP3,4184.0,126.137906,23.618215,64.0,88.0,96.0,122.0,170.0,198.0,254.0,3,0.07
CSBP4,4183.0,125.718623,23.366181,66.0,88.0,96.0,122.0,170.0,198.0,256.0,4,0.10
CSBP5,4171.0,125.562455,23.336731,64.0,88.0,96.0,122.0,170.0,198.0,254.0,16,0.38
CDBP1,4182.0,78.027021,13.606349,40.0,52.0,60.0,78.0,102.0,116.0,162.0,5,0.12
CDBP2,4183.0,78.266316,13.482345,40.0,52.0,60.0,78.0,102.0,114.0,160.0,4,0.10
CDBP3,4184.0,78.299474,13.525423,40.0,52.0,60.0,78.0,102.0,116.0,168.0,3,0.07
CDBP4,4183.0,78.237628,13.560585,40.0,52.0,58.0,76.0,102.0,116.0,168.0,4,0.10
CDBP5,4169.0,78.253778,13.591055,22.0,52.0,60.0,78.0,102.0,114.0,168.0,18,0.43


#### 4. Outcome Definition

Construct the Stage 3 hypertension outcome from the five clinic BP measurements and current antihypertensive drug therapy (`HTN_DR`).

In [45]:
# Calculate mean clinic SBP and DBP only when all five readings are available

bp_complete = nigeria[bp_vars].notna().all(axis=1)

nigeria["MEAN_SBP"] = nigeria[
    ["CSBP1", "CSBP2", "CSBP3", "CSBP4", "CSBP5"]
].mean(axis=1, skipna=False)

nigeria["MEAN_DBP"] = nigeria[
    ["CDBP1", "CDBP2", "CDBP3", "CDBP4", "CDBP5"]
].mean(axis=1, skipna=False)

print(f"Complete 5-reading BP: {bp_complete.sum():,}")
print(f"Incomplete 5-reading BP: {(~bp_complete).sum():,}")

Complete 5-reading BP: 4,167
Incomplete 5-reading BP: 20


In [ ]:
# Interpret missing HTN_DR as "No treatment" only where HTN=2,
# consistent with the observed questionnaire skip pattern.

htn_dr_outcome = nigeria["HTN_DR"].copy()

skip_implied_no = (
    nigeria["HTN"].eq(2) &
    nigeria["HTN_DR"].isna()    
)

htn_dr_outcome.loc[skip_implied_no] = 2

In [48]:
# Stage 3 hypertension outcome:
# measured hypertension OR current antihypertensive treatment.

measured_htn = (
    bp_complete &
    (
        (nigeria["MEAN_SBP"] >= 140) |
        (nigeria["MEAN_DBP"] >= 90)
    )
)

treated_htn = htn_dr_outcome.eq(1)

nigeria["HTN_STAGE3"] = np.where(
    measured_htn | treated_htn,
    1,
    np.where(
        bp_complete & ~measured_htn & htn_dr_outcome.notna(),
        0,
        np.nan
    )
)

In [49]:
# Outcome summary
outcome_summary = nigeria["HTN_STAGE3"].value_counts(dropna=False).sort_index()

print(outcome_summary)
print("\nPrevalence among classified participants:")
print(
    nigeria["HTN_STAGE3"].mean(skipna=True).round(4)
)

HTN_STAGE3
0.0    1984
1.0    1356
NaN     847
Name: count, dtype: int64

Prevalence among classified participants:
0.406


In [50]:
# Examine the sources of positive Stage 3 hypertension classifications.

print("Measured hypertension only:",
      (measured_htn & ~treated_htn).sum())

print("Treatment positive:",
      treated_htn.sum())

print("Both measured and treatment positive:",
      (measured_htn & treated_htn).sum())

print("Indeterminate outcome:",
      nigeria["HTN_STAGE3"].isna().sum())


Measured hypertension only: 861
Treatment positive: 495
Both measured and treatment positive: 319
Indeterminate outcome: 847
